In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import shutil
shutil.copy('/content/drive/MyDrive/DOUTORADO/DATASETS/Weather_2025_v4-1.csv', 'Weather_2025_v4-1.csv')

'Weather_2025_v4-1.csv'

In [ ]:
import pandas as pd

# Caminho do arquivo
path = "Weather_2025_v4-1.csv"

# Leitura do CSV
df = pd.read_csv(path)

# Visão inicial
df.head()


,Unnamed: 0,location_id,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,weather_code,pressure_msl,surface_pressure,cloud_cover,wind_speed_10m,wind_speed_100m,soil_temperature_0_to_7cm,soil_temperature_7_to_28cm,location
0,0,0,2025-01-01T00:00,22.9,86,20.4,24.5,0.0,0.0,0,1010.8,1002.7,5,16.4,28.0,24.4,24.3,86a901287ffffff
1,1,0,2025-01-01T01:00,22.2,88,20.1,23.7,0.0,0.0,0,1011.4,1003.3,3,16.4,28.5,23.2,24.2,86a901287ffffff
2,2,0,2025-01-01T02:00,22.0,90,20.2,23.7,0.0,0.0,0,1011.4,1003.2,2,15.4,27.2,22.9,24.1,86a901287ffffff
3,3,0,2025-01-01T03:00,21.8,92,20.4,23.8,0.0,0.0,0,1011.2,1003.0,3,14.1,25.4,22.6,24.0,86a901287ffffff
4,4,0,2025-01-01T04:00,21.8,92,20.4,23.8,0.0,0.0,0,1010.7,1002.5,0,13.9,25.2,22.5,23.9,86a901287ffffff


apparent_temperature >= 30°C  → calor desconfortável
apparent_temperature <= 10°C  → frio desconfortável

rain > 0.1 mm
OU
precipitation > 0.1


wind_speed >= 8 m/s


(apparent_temperature - temperature_2m) >= 3°C


Δ apparent_temperature >= 4°C em 3h
OU
Δ cloud_cover >= 30%


In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df = pd.read_csv("Weather_2025_v4-1.csv")

df["time"] = pd.to_datetime(df["time"], utc=True)
df = df.sort_values(["location_id", "time"])


#Aplicando o desconforto térmico

**Frio**

Muito frio: temp ≤ 10

Frio desconfortável:

10 < temp ≤ 15 e umidade ≥ 70



---


**Confortável**

18 ≤ temp ≤ 26

40 ≤ umidade ≤ 70


---



**Calor**

Calor seco:

26 < temp < 30 e umidade < 60

Calor úmido (desconforto forte):

temp ≥ 26 e umidade ≥ 60


---



**Muito calor:**

temp ≥ 32 (independente da umidade)


In [ ]:
def thermal_discomfort_level(temp, rh):
    # Frio
    if temp <= 10:
        return 3  # muito frio
    if temp <= 15 and rh >= 70:
        return 2  # frio desconfortável

    # Confortável
    if 18 <= temp <= 26 and 40 <= rh <= 70:
        return 0  # confortável

    # Calor
    if temp >= 32:
        return 3  # muito calor
    if temp >= 26 and rh >= 60:
        return 2  # calor úmido
    if temp >= 26:
        return 1  # calor leve

    return 0


In [ ]:
df["thermal_discomfort_level"] = df.apply(
    lambda r: thermal_discomfort_level(
        r["temperature_2m"],
        r["relative_humidity_2m"]
    ),
    axis=1
)

In [ ]:
df["thermal_discomfort"] = (
    # Frio extremo
    (df["temperature_2m"] <= 10) |

    # Frio úmido
    ((df["temperature_2m"] > 10) & (df["temperature_2m"] <= 15) & (df["relative_humidity_2m"] >= 70)) |

    # Calor úmido
    ((df["temperature_2m"] >= 26) & (df["relative_humidity_2m"] >= 60)) |

    # Muito calor
    (df["temperature_2m"] >= 32)
).astype(float)


In [ ]:
"""df["thermal_discomfort"] = (
    (df["temperature_2m"] >= 28.0) |
    (df["temperature_2m"] <= 15.0)
).astype(float)"""

'df["thermal_discomfort"] = (\n    (df["temperature_2m"] >= 28.0) |\n    (df["temperature_2m"] <= 15.0)\n).astype(float)'

In [ ]:
df["rain_flag"] = (
    (df["rain"] > 0.1) |
    (df["precipitation"] > 0.1)
).astype(int)


In [ ]:
df["wind_flag"] = (df["wind_speed_100m"] >= 8).astype(int)


In [ ]:
"""#Abafamento
df["mugginess"] = (
    (df["apparent_temperature"] - df["temperature_2m"]) >= 3
).astype(int)"""


'#Abafamento\ndf["mugginess"] = (\n    (df["apparent_temperature"] - df["temperature_2m"]) >= 3\n).astype(int)'

In [ ]:
#Instabilidade
df["temp_change_3h"] = (
    df.groupby("location_id")["apparent_temperature"]
      .diff(3)
      .abs()
)

df["cloud_change_3h"] = (
    df.groupby("location_id")["cloud_cover"]
      .diff(3)
      .abs()
)

df["instability"] = (
    (df["temp_change_3h"] >= 4) |
    (df["cloud_change_3h"] >= 30)
).astype(int)


In [ ]:
#Janela de 3 horas
WINDOW = 3

rolling_cols = [
    "thermal_discomfort",
    "rain_flag",
    "wind_flag",
    "thermal_discomfort_level",
    "instability"
]

for col in rolling_cols:
    df[f"{col}_3h"] = (
        df.groupby("location_id")[col]
          .rolling(WINDOW, min_periods=1)
          .max()
          .reset_index(level=0, drop=True)
    )


In [ ]:
#Indice de propensão de chamada
df["ride_hailing_favorable"] = (
    (df["thermal_discomfort_3h"] == 1) |
    (df["rain_flag_3h"] == 1) |
    (df["wind_flag_3h"] == 1) |
    (df["instability_3h"] == 1) |
    (df["thermal_discomfort_level_3h"] != 0)
).astype(int)


In [ ]:
df["ride_hailing_score"] = (
    2 * df["rain_flag_3h"] +
    1.5 * df["thermal_discomfort_3h"] +
    1.0 * df["wind_flag_3h"] +
    1.0 * df["instability_3h"] +
    0.5 * df["thermal_discomfort_level_3h"]
)


In [ ]:
cols = [
    "time",
    "location_id",
    "ride_hailing_favorable",
    "ride_hailing_score",
    "rain_flag_3h",
    "thermal_discomfort_3h",
    "wind_flag_3h",
    "instability_3h",
    "thermal_discomfort_level",
]

print(df[cols].head(20))


                        time  location_id  ride_hailing_favorable  \
0  2025-01-01 00:00:00+00:00            0                       1   
1  2025-01-01 01:00:00+00:00            0                       1   
2  2025-01-01 02:00:00+00:00            0                       1   
3  2025-01-01 03:00:00+00:00            0                       1   
4  2025-01-01 04:00:00+00:00            0                       1   
5  2025-01-01 05:00:00+00:00            0                       1   
6  2025-01-01 06:00:00+00:00            0                       1   
7  2025-01-01 07:00:00+00:00            0                       1   
8  2025-01-01 08:00:00+00:00            0                       1   
9  2025-01-01 09:00:00+00:00            0                       1   
10 2025-01-01 10:00:00+00:00            0                       1   
11 2025-01-01 11:00:00+00:00            0                       1   
12 2025-01-01 12:00:00+00:00            0                       1   
13 2025-01-01 13:00:00+00:00      

In [ ]:
print(
    df[df["ride_hailing_favorable"] == 1]
    [cols]
    .head(50)
)


                        time  location_id  ride_hailing_favorable  \
0  2025-01-01 00:00:00+00:00            0                       1   
1  2025-01-01 01:00:00+00:00            0                       1   
2  2025-01-01 02:00:00+00:00            0                       1   
3  2025-01-01 03:00:00+00:00            0                       1   
4  2025-01-01 04:00:00+00:00            0                       1   
5  2025-01-01 05:00:00+00:00            0                       1   
6  2025-01-01 06:00:00+00:00            0                       1   
7  2025-01-01 07:00:00+00:00            0                       1   
8  2025-01-01 08:00:00+00:00            0                       1   
9  2025-01-01 09:00:00+00:00            0                       1   
10 2025-01-01 10:00:00+00:00            0                       1   
11 2025-01-01 11:00:00+00:00            0                       1   
12 2025-01-01 12:00:00+00:00            0                       1   
13 2025-01-01 13:00:00+00:00      

In [ ]:
print(df["ride_hailing_favorable"].value_counts())


ride_hailing_favorable
1    253084
0      1652
Name: count, dtype: int64


In [ ]:
df["hour"] = df["time"].dt.hour

print(
    df.groupby("hour")["ride_hailing_score"]
      .mean()
      .round(2)
)


hour
0     2.85
1     2.73
2     2.58
3     2.47
4     2.47
5     2.50
6     2.52
7     2.56
8     2.60
9     2.61
10    2.65
11    2.83
12    3.12
13    3.31
14    3.28
15    3.14
16    3.05
17    2.97
18    2.91
19    2.86
20    2.99
21    3.18
22    3.15
23    3.02
Name: ride_hailing_score, dtype: float64


In [ ]:
from IPython.display import display

display(df[cols].head(2000))


,time,location_id,ride_hailing_favorable,ride_hailing_score,rain_flag_3h,thermal_discomfort_3h,wind_flag_3h,instability_3h,thermal_discomfort_level
0,2025-01-01 00:00:00+00:00,0,1,1.0,0.0,0.0,1.0,0.0,0
1,2025-01-01 01:00:00+00:00,0,1,1.0,0.0,0.0,1.0,0.0,0
2,2025-01-01 02:00:00+00:00,0,1,1.0,0.0,0.0,1.0,0.0,0
3,2025-01-01 03:00:00+00:00,0,1,1.0,0.0,0.0,1.0,0.0,0
4,2025-01-01 04:00:00+00:00,0,1,1.0,0.0,0.0,1.0,0.0,0
...,...,...,...,...,...,...,...,...,...
1995,2025-03-25 03:00:00+00:00,0,1,4.0,1.0,0.0,1.0,1.0,0
1996,2025-03-25 04:00:00+00:00,0,1,4.0,1.0,0.0,1.0,1.0,0
1997,2025-03-25 05:00:00+00:00,0,1,1.0,0.0,0.0,1.0,0.0,0
1998,2025-03-25 06:00:00+00:00,0,1,2.0,0.0,0.0,1.0,1.0,0


In [ ]:
df["time"] = df["time"].dt.tz_convert('America/Sao_Paulo')
print(df[cols].head(20))

                        time  location_id  ride_hailing_favorable  \
0  2024-12-31 21:00:00-03:00            0                       1   
1  2024-12-31 22:00:00-03:00            0                       1   
2  2024-12-31 23:00:00-03:00            0                       1   
3  2025-01-01 00:00:00-03:00            0                       1   
4  2025-01-01 01:00:00-03:00            0                       1   
5  2025-01-01 02:00:00-03:00            0                       1   
6  2025-01-01 03:00:00-03:00            0                       1   
7  2025-01-01 04:00:00-03:00            0                       1   
8  2025-01-01 05:00:00-03:00            0                       1   
9  2025-01-01 06:00:00-03:00            0                       1   
10 2025-01-01 07:00:00-03:00            0                       1   
11 2025-01-01 08:00:00-03:00            0                       1   
12 2025-01-01 09:00:00-03:00            0                       1   
13 2025-01-01 10:00:00-03:00      

In [ ]:
display(df.head(2000))

,Unnamed: 0,location_id,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,weather_code,...,cloud_change_3h,instability,thermal_discomfort_3h,rain_flag_3h,wind_flag_3h,thermal_discomfort_level_3h,instability_3h,ride_hailing_favorable,ride_hailing_score,hour
0,0,0,2024-12-31 21:00:00-03:00,22.9,86,20.4,24.5,0.0,0.0,0,...,NaN,0,0.0,0.0,1.0,0.0,0.0,1,1.0,0
1,1,0,2024-12-31 22:00:00-03:00,22.2,88,20.1,23.7,0.0,0.0,0,...,NaN,0,0.0,0.0,1.0,0.0,0.0,1,1.0,1
2,2,0,2024-12-31 23:00:00-03:00,22.0,90,20.2,23.7,0.0,0.0,0,...,NaN,0,0.0,0.0,1.0,0.0,0.0,1,1.0,2
3,3,0,2025-01-01 00:00:00-03:00,21.8,92,20.4,23.8,0.0,0.0,0,...,2.0,0,0.0,0.0,1.0,0.0,0.0,1,1.0,3
4,4,0,2025-01-01 01:00:00-03:00,21.8,92,20.4,23.8,0.0,0.0,0,...,3.0,0,0.0,0.0,1.0,0.0,0.0,1,1.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1995,0,2025-03-25 00:00:00-03:00,22.9,84,20.1,26.0,0.0,0.0,0,...,27.0,0,0.0,1.0,1.0,0.0,1.0,1,4.0,3
1996,1996,0,2025-03-25 01:00:00-03:00,22.5,86,20.1,25.6,0.0,0.0,3,...,2.0,0,0.0,1.0,1.0,0.0,1.0,1,4.0,4
1997,1997,0,2025-03-25 02:00:00-03:00,22.4,89,20.5,25.7,0.0,0.0,3,...,7.0,0,0.0,0.0,1.0,0.0,0.0,1,1.0,5
1998,1998,0,2025-03-25 03:00:00-03:00,22.4,91,20.8,26.0,0.0,0.0,3,...,66.0,1,0.0,0.0,1.0,0.0,1.0,1,2.0,6


Transformando H3 de RES 6 para H3 de RES 12

In [ ]:
!pip install h3

import h3
from h3 import LatLngPoly




In [ ]:
h3_res5_list = df['location'].unique()


all_res12 = set()

for h in h3_res5_list:
    boundary = h3.cell_to_boundary(h)
    poly = LatLngPoly(boundary)
    all_res12.update(h3.polygon_to_cells(poly, 12))

len(all_res12)
print(all_res12)

Buffered data was truncated after reaching the output size limit.

In [ ]:
df.to_csv("Weather_enhanced.csv")
import shutil
shutil.copy('Weather_enhanced.csv','/content/drive/MyDrive/DOUTORADO/DATASETS/DATASETS_PRONTOS/Weather_enhanced.csv')

KeyboardInterrupt: 